## Mounting the asset directory containing dataset

In [ ]:
import os

local_assets_b = False

if local_assets_b:
  assets_dir = "/content/assets/P2/"

  if not os.path.isdir(assets_dir):
    assert os.path.isfile("assets.zip")
    os.system("unzip assets.zip")
else:
  from google.colab import drive
  drive.mount('/content/drive')
  assets_dir = '/content/drive/MyDrive/IK/CV1/assets'

## VGG


### What is VGG?
VGG (Visual Geometry Group) is a popular deep convolutional neural network (CNN) architecture developed by the Visual Geometry Group at the University of Oxford. VGGNet is known for its simplicity and effectiveness in image classification tasks. It consists of multiple convolutional layers followed by fully connected layers. The most common variant, VGG-16, has 16 layers, including 13 convolutional layers and 3 fully connected layers. VGGNet has achieved impressive results on various image classification benchmarks, including the ImageNet challenge.

<center><img src="https://production-media.paperswithcode.com/methods/vgg_7mT4DML.png"/></center>

In [ ]:
# Import the required modules
import torch
import torch.nn as nn

In [ ]:
class VGG16(nn.Module):
    def __init__(self, num_classes=1000):
        super(VGG16, self).__init__()

        # Feature extraction layers (Convolutional + ReLU + MaxPooling)
        self.features = nn.Sequential(
            # Block 1: Extract low-level features (edges, textures)
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),  # Conv1_1
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True), # Conv1_2
            nn.MaxPool2d(kernel_size=2, stride=2),  # Downsamples feature maps (reduces H, W by 2)

            # Block 2: Extract higher-level patterns (simple shapes, colors)
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),  # Conv2_1
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True), # Conv2_2
            nn.MaxPool2d(kernel_size=2, stride=2),  # Further reduces spatial size

            # Block 3: Learn complex shapes and object parts
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),  # Conv3_1
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),  # Conv3_2
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),  # Conv3_3
            nn.MaxPool2d(kernel_size=2, stride=2),  # Reduce feature map size

            # Block 4: Learn even more abstract representations (e.g., full object shapes)
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),  # Conv4_1
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),  # Conv4_2
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),  # Conv4_3
            nn.MaxPool2d(kernel_size=2, stride=2),  # Reduce spatial size

            # Block 5: Detect complex object features (e.g., eyes, wheels, ears)
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),  # Conv5_1
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),  # Conv5_2
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),  # Conv5_3
            nn.MaxPool2d(kernel_size=2, stride=2)  # Final downsampling
        )

        # Fully connected classification layers
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096), nn.ReLU(True), nn.Dropout(),  # FC1: Learn abstract relationships
            nn.Linear(4096, 4096), nn.ReLU(True), nn.Dropout(),          # FC2: Further abstraction
            nn.Linear(4096, num_classes)  # FC3: Final classification layer
        )

    def forward(self, x):
        layer_outputs = []  # To store intermediate outputs for analysis/debugging

        # Forward pass through feature extractor
        for layer in self.features:
            x = layer(x)  # Apply each layer sequentially
            layer_outputs.append(x)  # Store feature maps at each stage

        # Flatten the output from the feature extractor
        x = x.view(x.size(0), -1)  # Convert from 3D (C, H, W) to 1D for fully connected layers

        # Forward pass through classifier
        for layer in self.classifier:
            x = layer(x)
            layer_outputs.append(x)  # Store outputs from FC layers

        return x, layer_outputs  # Return final output and all intermediate layers


### Code Explanation:
### What Happens in Each Block?
- **Block 1 (Conv + ReLU + MaxPool)**
  - Detects basic features like edges, textures, and color patterns.
  - MaxPooling reduces feature map size by half.
- **Block 2 (Conv + ReLU + MaxPool)**
  - Learns more complex shapes and simple object parts.
  - Further reduces the spatial size.
- **Block 3 (Conv + ReLU + Conv + ReLU + Conv + ReLU + MaxPool)**
  - Detects more complex structures such as object parts (e.g., eyes, wheels).
  - Deeper convolutional layers capture richer feature hierarchies.
- **Block 4 (Conv + ReLU + Conv + ReLU + Conv + ReLU + MaxPool)**
  - Abstracts larger object structures.
  - Helps differentiate between different object classes.
- **Block 5 (Conv + ReLU + Conv + ReLU + Conv + ReLU + MaxPool)**
  - Captures high-level object representations.
  - Final downsampling before fully connected layers.
- **Fully Connected Layers (FC1 → FC2 → FC3)**
  - FC1 (4096 neurons): Learns relationships between extracted features.
  - FC2 (4096 neurons): Further abstraction for decision-making.
  - FC3 (Output layer): Final classification into num_classes.


In [ ]:
# Initialise the model
vgg_model = VGG16(num_classes=10)  # Doing for 10 classes
print(vgg_model)

In [ ]:
# Calculate the total number of parameters in the model
total_params = sum(p.numel() for p in vgg_model.parameters())

# Print the total parameter count
print(f"Total Parameters: {total_params}")

# Each parameter (float32) takes 4 bytes of memory.
# Convert total parameters to MB: (total parameters * 4 bytes per parameter) / (1024 * 1024)
print(f'Total Model size in MB: {total_params * 4 / (1024 * 1024)}')


### Code Explanation:
- **p.numel():**
numel() returns the total number of elements (parameters) in a tensor.
for p in vgg_model.parameters() iterates over all model parameters (weights & biases).
- **Summing Up Parameters:**
The sum(...) computes the total count of all parameters in the model.
Memory Calculation:
Each parameter (weight/bias) is a float32 (4 bytes).
- **Convert to MB:** Divide by (1024 * 1024) to convert from bytes to megabytes (MB).


In [ ]:
# Create a random input tensor with shape (1, 3, 224, 224)
input_tensor = torch.rand(1, 3, 224, 224)

In [ ]:
input_tensor

In [ ]:
input_tensor = torch.rand(1, 3, 224, 224)
# Get intermediate layer outputs
x, layer_outputs = vgg_model(input_tensor)

# Print layer outputs
for i, output in enumerate(layer_outputs):
    print(f"Output after layer {i + 1}: {output.shape}")

print(f"Final layer output:  {x.shape}")

## ResNet



### Residual Networks (ResNet)

ResNet, short for Residual Network, is a specific type of neural network that was introduced in 2015 by Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun in their paper “Deep Residual Learning for Image Recognition”. The ResNet models were extremely successful, winning 1st place in the ILSVRC 2015 classification competition with a top-5 error rate of 3.57%. ResNet is a fully convolutional neural network with an encoder-decoder structure that has been adapted to incorporate other convolutional neural network architecture designs. The architecture of ResNet can be broadly thought of as a deep neural network with skip connections between layers that add the outputs from previous layers to the outputs of stacked layers. This results in the ability to train much deeper neural networks without running into the vanishing gradient problem. ResNet has many variants that run on the same concept but have different numbers of layers. ResNet50 is used to denote the variant that can work with 50 neural network layers. ResNet has significantly enhanced the performance of neural networks with more layers and has been used in various domains, including computer vision and biomedical imaging.

<center><img src="https://www.researchgate.net/publication/349646156/figure/fig4/AS:995806349897731@1614430143429/The-architecture-of-ResNet-50-vd-a-Stem-block-b-Stage1-Block1-c-Stage1-Block2.png"/></center>



## Designing ResNet


Reference:
[1] Kaiming He, Xiangyu Zhang, Shaoqing Ren, Jian Sun
    Deep Residual Learning for Image Recognition. arXiv:1512.03385



### Residual Layer

As discussed, after two convolution operations, input of those 2 convolution is added to their output. That is what we are implementing in the below code.


* Each residual block consists of 2 convolution operations
* Convolution -> batch norm -> Relu
* Except if downsampling has to be applied then the 2nd conv layer's output is added to input before applying ReLU.

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class ResidualLayer(nn.Module):
    def __init__(self, inp_c, out_c, stride=1, downsample=None):
        """
        Residual Layer for ResNet-like architectures.

        Args:
        - inp_c (int): Number of input channels.
        - out_c (int): Number of output channels.
        - stride (int, optional): Stride value for the first convolution. Defaults to 1.
        - downsample (nn.Module, optional): Downsampling layer if input and output dimensions differ. Defaults to None.
        """
        super().__init__()

        # First convolution layer: Applies a 3x3 convolution with optional stride (for downsampling)
        self.conv1 = nn.Conv2d(inp_c, out_c, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)  # Normalizes output to improve stability
        self.relu = nn.ReLU(inplace=True)  # In-place ReLU for memory efficiency

        # Second convolution layer: Maintains feature size (stride=1)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)  # Batch normalization

        # Downsampling layer (optional): Used when input and output shapes mismatch
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        """
        Forward pass of the residual block.

        Args:
        - x (Tensor): Input feature map.

        Returns:
        - Tensor: Output feature map after applying residual connections.
        """
        identity = x  # Store the original input for the residual connection

        # First convolution + batch normalization + ReLU activation
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        # Second convolution + batch normalization
        out = self.conv2(out)
        out = self.bn2(out)

        # Apply downsampling if needed (to match dimensions)
        if self.downsample is not None:
            identity = self.downsample(identity)  # Adjust input shape

        # Add the residual (skip) connection
        out += identity

        # Final ReLU activation
        out = self.relu(out)

        return out


### Code Explanation:
- **First Convolution + BatchNorm + ReLU:**
  - Extracts low-level features such as edges, textures, and small patterns.
  - Stride controls downsampling:
    - If stride = 1, keeps the spatial dimensions the same.
    - If stride > 1, reduces the height/width (downsampling).
  - Batch Normalization helps stabilize training and normalizes activations.
  - ReLU Activation introduces non-linearity, making the model capable of learning complex patterns.
- **Second Convolution + BatchNorm:**
  - Further refines extracted features from the first convolution.
  - Maintains the same spatial size (stride=1).
  - Batch Normalization again ensures stable activations and prevents overfitting.
- **Downsampling (Optional):**
  - If input and output feature maps do not match in shape, downsample is used.
  - Usually a 1x1 convolution is applied in downsample to align dimensions when:
    - inp_c ≠ out_c
    - stride > 1 (reducing feature map size)
- **Forward Pass – Applying First Convolution, BatchNorm, and ReLU:**
  - First convolution extracts features.
  - Batch normalization normalizes activations for better stability.
  - ReLU adds non-linearity and prevents vanishing gradients.
- **Applying Second Convolution and BatchNorm:**
  - Further processes the extracted features from the first block.
  - No ReLU here yet because we need to add the residual connection first.
- **Residual Connection (Skip Connection):**
  - Handles dimensional mismatch:
    - If downsample exists, it transforms x to match out in shape.
  - Skip connection adds the original input (x) to the transformed output (out).
    - This helps prevent vanishing gradients in deep networks.
    - Enables better gradient flow and allows deeper networks to be trained.
- **Final Activation:**
  - Applies ReLU after the residual connection.
  - Ensures that the output maintains non-linearity and useful features.
  - The final processed feature map is returned.

In [ ]:
ResidualLayer(64, 128)  # Creates a residual block that takes 64 input channels and outputs 128 channels

### make_block method

In [ ]:
def make_block(residual_layer, inp_c, out_c, num_residual_layers_in_block, stride=1):
    """
    Creates a block of multiple residual layers (used in ResNet-like architectures).

    Args:
    - residual_layer (nn.Module): The residual block class (e.g., ResidualLayer).
    - inp_c (int): Number of input channels.
    - out_c (int): Number of output channels.
    - num_residual_layers_in_block (int): Number of residual layers in this block.
    - stride (int, optional): Stride for the first residual layer. Default is 1 (no downsampling).

    Returns:
    - nn.Sequential: A sequential container of residual layers.
    """

    downsample = None  # Default to no downsampling

    # If stride > 1 OR input/output channels are different, we need a downsampling layer
    if stride != 1 or out_c != inp_c:
        downsample = nn.Sequential(
            nn.Conv2d(inp_c, out_c, kernel_size=1, stride=stride, bias=False),  # 1x1 Conv for downsampling
            nn.BatchNorm2d(out_c)  # BatchNorm to stabilize training
        )

    layers = []  # List to store residual layers

    # First residual layer in the block (handles downsampling if required)
    layers.append(residual_layer(inp_c, out_c, stride, downsample))

    inp_c = out_c  # Update input channel count for the next residual layers

    # Add remaining residual layers (without downsampling)
    for _ in range(1, num_residual_layers_in_block):
        layers.append(residual_layer(inp_c, out_c))  # Subsequent layers keep same in/out channels

    return nn.Sequential(*layers)  # Return as a Sequential block


One of the defining characteristics of ResNet is its deep and repetitive structure, where residual blocks are stacked together in a highly modular fashion. To streamline the creation of these blocks and facilitate the assembly of the network, the `make_block` function is employed.

**Function Purpose**:
- The `make_block` function is instrumental in constructing the core building blocks of a ResNet architecture. These blocks are stacked repeatedly to form the complete network.

**Parameters**:
- `residual_layer`: This parameter corresponds to the specific type of residual block used in ResNet. Residual blocks contain skip connections and enable the network to learn residual functions.
- `inp_c`: It denotes the number of input channels (or feature maps) entering the layer, which depends on the network architecture and the previous layer.
- `out_c`: This parameter specifies the number of output channels (feature maps) that the layer should produce. It's a crucial factor in determining the network's capacity.
- `num_residual_layers_in_block`: ResNet architectures consist of multiple residual blocks within a single layer. The `num_residual_layers_in_block` parameter indicates how many times the `residual_layer` type should be repeated within the layer.
- `stride`: Stride controls the spatial downsampling applied within the layer. It determines how the layer processes input data and affects the spatial dimensions of the feature maps.

**Downsampling**:
- ResNet architectures often involve spatial downsampling to reduce feature map dimensions. Downsampling typically occurs when `stride` is not equal to 1 or when the number of output channels `out_c` differs from the number of input channels `inp_c`. In such cases, a downsampling mechanism is created using a 1x1 convolutional layer followed by batch normalization. This mechanism is added as a `downsample` module to match the dimensions.

**Layer Construction**:
- The `make_block` function initializes a list called `layers` to hold the sequence of residual blocks.
- The initial block, which may include downsampling if required, is appended to the `layers` list.
- `inp_c` is updated to match `out_c` because subsequent residual blocks within the same layer expect `out_c` channels as input.
- Additional blocks, as specified by the `num_residual_layers_in_block` parameter, are appended to the `layers` list. These blocks typically do not involve downsampling.

**Sequential Module**:
- Finally, all the residual blocks within the `layers` list are grouped together into a single sequential module using PyTorch's `nn.Sequential`. This sequential module encapsulates the entire layer, allowing it to be easily integrated into the broader ResNet architecture.

In [ ]:
make_block(ResidualLayer, inp_c=64, out_c=64, num_residual_layers_in_block=3, stride=2)
# Creates a block of 3 residual layers where:
# - Input has 64 channels, output has 64 channels (no change in channel depth).
# - The first residual layer uses stride=2, downsampling the spatial dimensions (H, W reduced by half).
# - A downsampling layer (1x1 conv + BatchNorm) is applied to the skip connection in the first residual layer.
# - The remaining two residual layers maintain the same spatial size and channel depth (64 → 64).
# - The block is returned as an nn.Sequential module containing these 3 residual layers.

### ResNet Model Class

In [ ]:
import torch
import torch.nn as nn

## ResNet Class
class ResNet(nn.Module):
    def __init__(self, residual_layer, layers, num_classes=10):
        """
        ResNet model implementation.

        Args:
        - residual_layer (nn.Module): Residual block class (e.g., ResidualLayer).
        - layers (list of int): Number of residual blocks in each layer.
        - num_classes (int): Number of output classes (default=10 for classification tasks like CIFAR-10).
        """
        super().__init__()

        # Initial input channels
        self.inp_c = 64  # The first convolution layer will output 64 channels.

        # First Convolutional Layer
        self.conv1 = nn.Conv2d(3, self.inp_c, kernel_size=7, stride=2, padding=3, bias=False)
        # - 7x7 kernel captures larger patterns from input images.
        # - Stride = 2 downsamples the input (224x224 → 112x112).
        # - Padding = 3 maintains proper spatial dimensions.

        self.bn1 = nn.BatchNorm2d(self.inp_c)  # Normalize feature maps.
        self.relu = nn.ReLU(inplace=True)  # In-place activation saves memory.

        # Max Pooling Layer
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        # - 3x3 max pooling further reduces spatial size (112x112 → 56x56).

        # Creating Residual Blocks
        self.layer1 = self.__make_block(residual_layer, 64, layers[0])  # Output: 56x56
        self.layer2 = self.__make_block(residual_layer, 128, layers[1], stride=2)  # Output: 28x28
        self.layer3 = self.__make_block(residual_layer, 256, layers[2], stride=2)  # Output: 14x14
        self.layer4 = self.__make_block(residual_layer, 512, layers[3], stride=2)  # Output: 7x7

        # Global Average Pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        # - Converts 7x7 feature maps into 1x1 feature maps (global pooling).

        # Fully Connected Layer for Classification
        self.fc = nn.Linear(512, num_classes)
        # - Final classification layer that maps feature vectors to `num_classes`.

    def __make_block(self, block, out_c, blocks, stride=1):
        """
        Creates a block of multiple residual layers.

        Args:
        - block (nn.Module): Residual block class.
        - out_c (int): Number of output channels.
        - blocks (int): Number of residual layers in this block.
        - stride (int, optional): Stride value for the first residual layer in this block. Default is 1.

        Returns:
        - nn.Sequential: A sequential container of residual layers.
        """
        downsample = None  # Default to no downsampling

        # If stride > 1 OR input/output channels are different, create a downsampling layer
        if stride != 1 or out_c != self.inp_c:
            downsample = nn.Sequential(
                nn.Conv2d(self.inp_c, out_c, kernel_size=1, stride=stride, bias=False),  # 1x1 conv for downsampling
                nn.BatchNorm2d(out_c)  # Normalize downsampled output
            )

        layers = []
        layers.append(block(self.inp_c, out_c, stride, downsample))  # First residual layer (handles downsampling)
        self.inp_c = out_c  # Update input channels for subsequent residual layers

        # Remaining residual layers (no downsampling)
        for _ in range(1, blocks):
            layers.append(block(self.inp_c, out_c))

        return nn.Sequential(*layers)  # Return as a Sequential block

    def forward(self, x):
        """
        Forward pass through the ResNet model.

        Args:
        - x (Tensor): Input image tensor.

        Returns:
        - Tensor: Final output predictions.
        - List: Intermediate feature maps from different layers.
        """
        layer_outputs = []  # Store intermediate feature maps for visualization/debugging.

        # Initial Convolution + BN + ReLU
        x = self.conv1(x)  # Input: (224x224) → Output: (112x112)
        layer_outputs.append(x)  # Store feature map

        x = self.bn1(x)  # Normalize
        x = self.relu(x)  # Apply ReLU activation

        # Max Pooling
        x = self.maxpool(x)  # Input: (112x112) → Output: (56x56)
        layer_outputs.append(x)

        # Residual Blocks
        x = self.layer1(x)  # Residual Block 1 → Output: (56x56)
        layer_outputs.append(x)

        x = self.layer2(x)  # Residual Block 2 → Output: (28x28)
        layer_outputs.append(x)

        x = self.layer3(x)  # Residual Block 3 → Output: (14x14)
        layer_outputs.append(x)

        x = self.layer4(x)  # Residual Block 4 → Output: (7x7)
        layer_outputs.append(x)

        # Global Average Pooling
        x = self.avgpool(x)  # Output: (1x1)
        layer_outputs.append(x)

        # Flatten feature maps into a vector for classification
        x = torch.flatten(x, 1)

        # Fully Connected Layer for Classification
        x = self.fc(x)
        layer_outputs.append(x)

        return x, layer_outputs  # Return final output and all intermediate layer activations


In [ ]:
# Define the ResNet-18 architecture
layers = [2, 2, 2, 2]
# - Specifies the number of residual layers in each block.
# - ResNet-18 consists of 4 blocks, with [2, 2, 2, 2] residual layers in each block.

# Instantiate the ResNet-18 model using the defined ResidualLayer
resnet_model = ResNet(ResidualLayer, layers)
# - Creates a ResNet-18 model with:
#   - 2 residual layers in Block 1 (64 channels)
#   - 2 residual layers in Block 2 (128 channels, stride=2)
#   - 2 residual layers in Block 3 (256 channels, stride=2)
#   - 2 residual layers in Block 4 (512 channels, stride=2)
# - Uses ResidualLayer as the basic block.
# - Default number of output classes = 10 (can be changed when needed).

# Model follows the standard ResNet-18 architecture, designed for tasks like ImageNet classification or CIFAR-10.


### Code Explanation:
### ResNet-18 Configuration:
- layers = [2,2,2,2] → Means each of the 4 residual blocks contains 2 residual layers.
- This structure matches ResNet-18, which has 18 layers in total (including the initial conv layers).
### Model Instantiation (ResNet(ResidualLayer, layers)):
- Uses ResidualLayer as the building block.
- Creates a lightweight yet deep neural network for image classification.
- Can be extended to deeper versions like ResNet-34, ResNet-50, etc. by changing layers.
This is the standard ResNet-18 model setup, optimized for feature extraction and classification.

In [ ]:
resnet_model

In [ ]:
# Calculate the total number of trainable parameters in the ResNet model
total_params = sum(p.numel() for p in resnet_model.parameters())

# Print the total number of parameters in the model
print(f"Total Parameters: {total_params}")

# Calculate the total model size in megabytes (MB)
# - Each parameter is stored as a 32-bit float (4 bytes)
# - Convert total bytes to MB: (total parameters * 4 bytes) / (1024 * 1024)
print(f"Total Model Size MB: {total_params * 4 / 1024 / 1024}")


In [ ]:
input_tensor = torch.rand(1, 3, 224, 224)
# Get intermediate layer outputs
x, layer_outputs = resnet_model(input_tensor)

# Print layer outputs
for i, output in enumerate(layer_outputs):
    print(f"Output after layer {i + 1}: {output.shape}")

print("Final layer output: ", x.shape)

## Training the Models

#### What dataset are we using?
We will be working on CIFAR-10 which is a widely used benchmark dataset in computer vision and machine learning. It consists of 60,000 small-sized color images (32x32 pixels) belonging to 10 different classes, with 6,000 images per class. The dataset is split into a training set of 50,000 images and a test set of 10,000 images. The classes in CIFAR-10 include common objects like airplanes, automobiles, birds, cats, deer, dogs, frogs, horses, ships, and trucks. CIFAR-10 serves as a good dataset for evaluating and benchmarking image classification models.

When it comes to transfer learning, VGG is often used as a backbone model. Its pre-trained weights, which have been learned on the large-scale ImageNet dataset, capture generic features like edges, textures, and shapes that are beneficial for various visual recognition tasks. By leveraging the pre-trained VGG model, we can fine-tune it on the CIFAR-10 dataset to perform image classification. The lower-level layers of VGG capture low-level features, such as edges and corners, while the higher-level layers learn more complex features. This enables VGG to extract meaningful representations from images and generalize well to new tasks with limited labeled data.

By fine-tuning VGG on the CIFAR-10 dataset, we can take advantage of the pre-trained weights and learn task-specific features for image classification. This approach is effective when the target task has a similar domain or visual characteristics as the source task on which VGG was pre-trained. Transfer learning with VGG can help achieve better performance on CIFAR-10 by leveraging the knowledge learned from ImageNet, even with a smaller dataset.


These are the classes present in CIFAR-10:

 `('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')`

 <center><img src="https://corochann.com/wp-content/uploads/2021/09/cifar10_plot.png"/></center>

### Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms

import numpy as np

In [ ]:
classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
# Import necessary libraries
import torchvision
import torchvision.transforms as transforms

# Define a transform to preprocess the data
transform = transforms.Compose([
    transforms.ToTensor()  # Converts images to PyTorch tensors and normalizes pixel values to [0,1]
])

# Load the CIFAR-10 training dataset
trainset = torchvision.datasets.CIFAR10(
    root='./data',  # Directory where the dataset will be stored/downloaded
    train=True,  # Load the training set (if False, loads the test set)
    download=True,  # Download the dataset if it is not already present
    transform=transform  # Apply the defined preprocessing transformations
)


### Image Transformation
We will be performing the following transformations:

*   Random Crop
*   Flip
*   Normalize data

In [ ]:
import torchvision.transforms as transforms

# Define a set of transformations for training data augmentation
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    # Randomly crops a 32x32 region from an image with a padding of 4 pixels.
    # Helps in generalization by allowing the model to learn from slightly varied versions of the same image.

    transforms.RandomHorizontalFlip(),
    # Randomly flips the image horizontally with a 50% probability.
    # Improves robustness to object orientations.

    # Additional transformations (commented out) can be used for further augmentation:
    # transforms.RandomVerticalFlip(),  # Randomly flips the image vertically.
    # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Adjusts brightness, contrast, etc.
    # transforms.GaussianBlur(kernel_size=(3,3)),  # Applies a Gaussian blur to the image.
    # transforms.RandomRotation(degrees=15),  # Rotates the image by up to ±15 degrees.

    transforms.ToTensor(),
    # Converts images to PyTorch tensors and normalizes pixel values from [0, 255] to [0, 1].

    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2434, 0.2616))
    # Normalizes each channel of the image (mean and std for CIFAR-10 dataset).
    # Helps in stabilizing training and improving convergence.
])


# np.mean(imagenet)
# np.std(image)
# Image -> 10 augmentations -> label

# 1 > 2> 3> 4

# 512, 512 - >.

Test time Transformations
* Normalization

In [ ]:
import torchvision.transforms as transforms

# Define a transformation pipeline for test data (no augmentation)
transform_test = transforms.Compose([
    transforms.ToTensor(),
    # Converts images to PyTorch tensors and normalizes pixel values from [0, 255] to [0, 1].

    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2434, 0.2616))
    # Normalizes each channel using the mean and standard deviation of the CIFAR-10 dataset.
    # Ensures consistency in data distribution between training and testing.
])


### Dataset

Datasets are the collections of your training, validation, and test data. They consist of input samples and their corresponding target labels (for supervised learning). In PyTorch, datasets are typically created using custom classes inheriting from `torch.utils.data.Dataset`. You load your data into this class, allowing easy access during training. PyTorch provides built-in datasets like MNIST, CIFAR-10, and ImageNet, but custom datasets can also be created to work with specific data.

In [ ]:
## Preparing Dataset

# Load the CIFAR-10 training dataset with transformations
trainset = torchvision.datasets.CIFAR10(
    root='./data',  # Directory where the dataset will be stored/downloaded
    train=True,  # Load the training set (if False, loads the test set)
    download=True,  # Download the dataset if it is not already present
    transform=transform_train  # Apply defined transformations (augmentation + normalization)
)

# Create a DataLoader for efficient data loading
trainloader = torch.utils.data.DataLoader(
    trainset,  # Dataset to load
    batch_size=128,  # Number of samples per batch (128 images at a time)
    shuffle=True,  # Shuffle the dataset at every epoch to improve generalization
    num_workers=2  # Number of parallel data-loading threads (speeds up data processing)
)

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

def imshow(img):
    """
    Displays a PyTorch image tensor after unnormalizing it.

    Args:
    - img (Tensor): Image tensor in the shape (C, H, W)

    Steps:
    1. Unnormalize the image by reversing the normalization transformation.
    2. Convert the tensor to a NumPy array for visualization.
    3. Transpose the dimensions from (C, H, W) to (H, W, C) for proper display.
    4. Display the image using Matplotlib.
    """

    img = img / 2 + 0.5  # Unnormalize (Reversing the normalization applied during training)

    npimg = img.numpy()  # Convert the tensor to a NumPy array

    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    # Transpose dimensions from (C, H, W) → (H, W, C) (Matplotlib expects this format)

    plt.show()  # Display the image

In [ ]:
# Get an iterator for the training data
dataiter = iter(trainloader)
# - Creates an iterable object for the trainloader to access batches sequentially.

images, labels = next(dataiter)
# - Retrieves the next batch of images and corresponding labels from the dataset.
# - `images` shape: (batch_size, channels, height, width), e.g., (128, 3, 32, 32).
# - `labels` contains the class indices for the images.

# Show images in a grid format
imshow(torchvision.utils.make_grid(images))
# - `make_grid(images)`: Arranges multiple images into a single grid for visualization.
# - Passes the grid to `imshow()` to display.

# Print labels corresponding to displayed images
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(len(labels))))
# - `classes[labels[j]]`: Converts label indices into actual class names.
# - `join(...)`: Formats labels into a single-line string for easy readability.
# - Displays class names of the batch images in order.

### Dataloaders

Dataloaders, are utilities that enable efficient data loading and batching. They take a dataset as input and allow users to define batch sizes, shuffle the data, and apply transformations to the samples. Dataloaders are especially useful when dealing with large datasets, as they enable the model to process data in small batches, reducing memory requirements and speeding up training. They are key components in PyTorch that facilitate data handling and preparation for machine learning tasks.

In [ ]:
# Load the CIFAR-10 test dataset with transformations
testset = torchvision.datasets.CIFAR10(
    root='./data',  # Directory where the dataset will be stored/downloaded
    train=False,  # Load the test set (train=False)
    download=True,  # Download the dataset if it is not already present
    transform=transform_test  # Apply defined transformations (normalization only, no augmentation)
)

# Create a DataLoader for efficient batch processing of the test dataset
testloader = torch.utils.data.DataLoader(
    testset,  # Dataset to load
    batch_size=256,  # Number of images per batch (256 images at a time)
    shuffle=True,  # Shuffle test data for better evaluation randomness
    num_workers=2  # Number of parallel data-loading threads (speeds up processing)
)

In [ ]:
len(trainloader), len(testset)%256

### Learning Rate

The learning rate is a hyperparameter that controls how much the model's parameters should be updated during training.

In [ ]:
lr = 0.01

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
device

### Loss Function

Loss functions measure the difference between the predicted output and the actual target values. Common loss functions include Cross-Entropy Loss for classification tasks and Mean Squared Error for regression tasks.

In [ ]:
# Define the loss function (criterion)
criterion = nn.CrossEntropyLoss()

### Code Explanation:
- `nn.CrossEntropyLoss()` is a commonly used loss function for multi-class classification problems.
- It computes the difference between the model's predicted class probabilities (logits) and the actual class labels.
- Automatically applies softmax to logits before calculating loss, so the model’s output should be raw scores (logits), not probabilities.


### Optimizer

Optimizers are algorithms that adjust the model's parameters during training to minimize the loss function. Common optimizers include SGD (Stochastic Gradient Descent), Adam, and RMSprop.

In [ ]:
# Define the optimizer for training ResNet
resnet_optimizer = optim.SGD(
    resnet_model.parameters(),  # Passes all model parameters to the optimizer for updates
    lr=1e-3,  # Learning rate (controls how much to adjust weights in each step)
    momentum=0.9,  # Momentum helps accelerate learning and smooth out updates
    weight_decay=5e-4  # L2 regularization (prevents overfitting by penalizing large weights)

### Code Explanation:
- `optim.SGD` → Uses Stochastic Gradient Descent (SGD), a popular optimization algorithm for deep learning.
- `lr=1e-3 (0.001)` → Learning rate, which determines the step size in weight updates.
- `momentum=0.9` → Helps accelerate gradient updates, preventing oscillations and improving convergence.
- `weight_decay=5e-4` → Implements L2 regularization, preventing large weights and reducing overfitting.

### Scheduler

A scheduler adjusts the learning rate dynamically during training, allowing fine-tuning.

Cosine Annealing: The learning rate starts high and is annealed down to a minimum value following a cosine curve. It helps the model explore the search space broadly at the beginning of training and then refine the search space as it converges.

T_max: This parameter defines the total number of iterations it takes to complete one cycle of the cosine function. The learning rate will follow a cosine curve for the first T_max iterations and then restart the cycle.

Here's a conceptual explanation:

At the start of training, the learning rate is relatively high, allowing the model to explore a larger area of the loss landscape.
As training progresses (over the T_max iterations), the learning rate decreases following a cosine curve.
When T_max iterations are completed, the learning rate is at its minimum.
The scheduler then restarts the cosine curve, and the learning rate starts to increase again, allowing the model to explore broadly for the next cycle.
This approach often helps models converge more efficiently by first exploring broadly and then refining their parameters as training progresses.

In [ ]:
# Define the learning rate scheduler for ResNet training
resnet_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    resnet_optimizer,  # The optimizer whose learning rate will be adjusted
    T_max=200  # Maximum number of epochs before completing one annealing cycle
)

Set the models in training mode

In [ ]:
resnet_model.train()

In [ ]:
# Move the ResNet model to the specified device (CPU or GPU)
resnet_model = resnet_model.to(device)


In [ ]:
def train_batch(epoch, model, optimizer):
    """
    Trains the model for one epoch using the training dataset.

    Args:
    - epoch (int): The current training epoch number.
    - model (nn.Module): The ResNet model to be trained.
    - optimizer (torch.optim): The optimizer used for updating model weights.

    Prints:
    - Training loss and accuracy for the current epoch.
    """

    print("Epoch", epoch)  # Print the current epoch number
    model.train()  # Set the model to training mode

    train_loss = 0  # Variable to accumulate total loss
    correct = 0  # Counter for correctly classified samples
    total = 0  # Counter for total samples processed

    # Iterate over the training batches
    for batch_idx, (input, targets) in enumerate(trainloader):
        # Move input and targets to the same device as the model (CPU/GPU)
        inputs, targets = input.to(device), targets.to(device)

        optimizer.zero_grad()  # Clear previous gradients
        outputs, _ = model(inputs)  # Forward pass: Get model predictions

        loss = criterion(outputs, targets)  # Compute loss using CrossEntropyLoss
        loss.backward()  # Backpropagation: Compute gradients
        optimizer.step()  # Update model parameters

        train_loss += loss.item()  # Accumulate total loss

        _, predicted = outputs.max(1)  # Get class predictions (argmax over class dimension)
        total += targets.size(0)  # Count total samples processed
        correct += predicted.eq(targets).sum().item()  # Count correct predictions

    # Print training statistics: Batch index, total loss, and accuracy
    print(batch_idx, len(trainloader),
          'Loss: %.3f | Acc: %.3f%% (%d/%d)'
          % (train_loss / (batch_idx + 1),  # Average loss per batch
             100. * correct / total,  # Compute accuracy percentage
             correct, total))  # Display correct/total samples


In [ ]:
def validate_batch(epoch, model):
    """
    Validates the model on the test dataset for one epoch.

    Args:
    - epoch (int): The current epoch number.
    - model (nn.Module): The trained model to be evaluated.

    Prints:
    - Validation loss and accuracy for the current epoch.
    """

    global best_acc  # Track the best accuracy achieved so far
    model.eval()  # Set the model to evaluation mode (disables dropout, batch norm updates)

    test_loss = 0  # Variable to accumulate total validation loss
    correct = 0  # Counter for correctly classified samples
    total = 0  # Counter for total samples processed

    with torch.no_grad():  # Disable gradient computation (reduces memory usage, speeds up validation)
        for batch_idx, (inputs, targets) in enumerate(testloader):
            # Move inputs and targets to the same device as the model (CPU/GPU)
            inputs, targets = inputs.to(device), targets.to(device)

            outputs, _ = model(inputs)  # Forward pass: Get model predictions
            loss = criterion(outputs, targets)  # Compute loss using CrossEntropyLoss

            test_loss += loss.item()  # Accumulate total loss
            _, predicted = outputs.max(1)  # Get predicted class (argmax over class dimension)
            total += targets.size(0)  # Count total samples processed
            correct += predicted.eq(targets).sum().item()  # Count correct predictions

    # Print validation statistics: Batch index, total loss, and accuracy
    print(batch_idx, len(testloader),
          'Loss: %.3f | Acc: %.3f%% (%d/%d)'
          % (test_loss / (batch_idx + 1),  # Average loss per batch
             100. * correct / total,  # Compute accuracy percentage
             correct, total))  # Display correct/total samples


In [ ]:
# Set the starting epoch
start_epoch = 0  # Typically 0 unless resuming from a checkpoint

# Train and validate the model for 20 epochs
for epoch in range(start_epoch, start_epoch + 20):

    # Train the model for one epoch using the training dataset
    train_batch(epoch, resnet_model, resnet_optimizer)

    # Validate the model on the test dataset after each training epoch
    validate_batch(epoch, resnet_model)

    # Adjust the learning rate based on the Cosine Annealing schedule
    resnet_scheduler.step()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Save the models

In [ ]:
model_path = 'resnet_model.pt'  # asset_dir + 'resnet_model.pt'
# Save the model's state dictionary to the specified path
#torch.save(resnet_model.state_dict(), model_path)

In [ ]:
!pwd

In [ ]:
assets_dir

In [ ]:
# Load the model
# assets_dir = '/content/drive/MyDrive/assets/P2/'
# file_path = assets_dir + 'resnet_model.pt'
assets_dir = "/content/drive/MyDrive/CV_revamp/CV_1_P2_assets/In_class/"
file_path = assets_dir + 'resnet_model.pt'
# NOTE: Need to run the cells for BasicBlock class, make_block method and ResNet class

print(file_path)
# Load the saved state dictionary
saved_state_dict = torch.load(file_path, map_location=torch.device('cpu'))

# Load the state dictionary into the model
resnet_model.load_state_dict(saved_state_dict)

# Set the model to evaluation mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet_model = resnet_model.to(device)
resnet_model.eval()

## Model Evaluation

The models performance is the same as the trained models

In [ ]:
epoch = 1
validate_batch(epoch, resnet_model)

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Define the transformations for loading the CIFAR-10 dataset
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2434, 0.2616))
])

# Load the CIFAR-10 test dataset
testset = torchvision.datasets.CIFAR10(root = './data', train = False, download = True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size = 256, shuffle = True, num_workers = 2)

In [ ]:
# Use the trained model from previous training
model = resnet_model  # Assign the trained ResNet model to `model`

# Mean and standard deviation used for normalization (CIFAR-10 dataset)
mean = [0.4914, 0.4822, 0.4465]
std = [0.2470, 0.2434, 0.2616]

# Evaluate the model on random test images and display results
for _ in range(10):  # Run evaluation on 10 random images

    # Get a random test image and its actual label
    data, target = next(iter(testloader))  # Fetch a batch from testloader

    # Get model's predictions for the batch
    output, _ = model(data.to(device))  # Forward pass through the model
    _, predicted = torch.max(output, 1)  # Get the class index with the highest probability

    # Extract the first image from the batch
    display_img = data[0]  # Select the first image for visualization

    # Unnormalize the image to convert it back to original pixel values
    unnormalized_image = display_img.clone()  # Create a copy to avoid modifying the original tensor
    for i in range(3):  # Iterate over the 3 color channels (RGB)
        unnormalized_image[i] = (unnormalized_image[i] * std[i]) + mean[i]  # Reverse normalization

    # Convert tensor to NumPy array and adjust shape for display
    plt.imshow(np.transpose(unnormalized_image.numpy(), (1, 2, 0)))  # Convert (C, H, W) → (H, W, C)

    # Set title with predicted and actual class labels
    plt.title(f'Predicted: {classes[predicted[0]]}, Actual: {classes[target[0]]}')

    plt.axis('off')  # Hide axis for better visualization
    plt.show()  # Display the image

## Exercise

You can try to train a VGG model and perform the same steps as performed for ResNet.

# Visualising the layers

## Visualizing different layers of VGG and ResNet

In the VGG and ResNet network, as the input image progresses through the convolutional layers, it undergoes a series of transformations and feature extractions. One fascinating aspect of VGG and ResNet is the ability to visualize the learned features at different convolutional layers, which provides insights into the network's inner workings.

By examining the feature maps of various convolutional layers, we can observe how the network progressively captures different levels of abstraction. In the early layers, such as the first few convolutional layers, the network tends to learn simple and low-level features like edges, corners, and textures. These features are more local and specific to certain image regions.

As we move deeper into the network, the feature maps become more complex and abstract. Higher-level layers capture more global and semantic information, focusing on object parts, shapes, and textures. These learned features are more robust and capable of representing more complex visual patterns.

Visualizing the intermediate layers of VGG helps us understand how the network gradually builds a hierarchy of features, with each layer refining and enhancing the representations learned in the previous layers. It also highlights the network's ability to transform raw pixel values into rich, hierarchical feature representations that enable accurate image classification and object detection.

By analyzing the visualizations of different conv layers, we gain valuable insights into the network's feature learning process, offering a glimpse into how deep convolutional neural networks extract and encode meaningful visual information from images. This understanding not only aids in interpreting the network's decisions but also provides a foundation for developing improved architectures and advancing the field of computer vision.






## VGG

### Importing the necessary modules

In [ ]:
# papers with code
# arxiv

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import matplotlib.pyplot as plt
from PIL import Image

### Load the Model

In [ ]:
# Load the pre-trained VGG model
model = models.vgg16(pretrained=True)
print(model)

### Extracting the Convolution Layers from VGG

In [ ]:
import torch.nn as nn

# We will save the convolutional layer weights in this list
model_weights = []

# We will store all the convolutional layers in this list
conv_layers = []

# Get all the model's children (layers and blocks) as a list
model_children = list(model.children())

# Iterate through the model's layers to extract convolutional layers
for i in range(len(model_children)):
    print("Block: ", i, " : ", type(model_children[i]))  # Print block number and its type

    # Check if the current block is a Sequential container (which contains multiple layers)
    if isinstance(model_children[i], torch.nn.modules.container.Sequential):

        # Iterate over each layer inside the Sequential block
        for layer_num, child in enumerate(model_children[i].children()):
            print("Layer: ", layer_num, " : ", type(child))  # Print layer number and its type

            # Check if the layer is a Convolutional layer
            if isinstance(child, nn.Conv2d):
                model_weights.append(child.weight)  # Save the layer's weights
                conv_layers.append(child)  # Store the convolutional layer itself

# Print the total number of convolutional layers found
print(f"Total convolution layers found: {len(conv_layers)}")


### Load the image

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms

# Load the image from the specified file path
image = Image.open(fp=assets_dir + 'dog_image.jpeg')  # Open the image using PIL

# Display the original image
plt.imshow(image)
plt.axis('off')  # Hide axis for better visualization
plt.show()

# Define transformations to preprocess the image before feeding it into VGG
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize the image to 224x224 pixels (required input size for VGG)
    transforms.ToTensor(),  # Convert the image to a PyTorch tensor with pixel values in [0,1]
])

# Apply the transformations to the image
image = transform(image)

# Print the shape before adding a batch dimension
print(f"Image shape before: {image.shape}")  # Expected output: (C, H, W) → (3, 224, 224)

# Add a batch dimension to match the expected input shape for VGG
image = image.unsqueeze(0)  # Changes shape from (3, 224, 224) to (1, 3, 224, 224)

# Print the shape after adding a batch dimension
print(f"Image shape after: {image.shape}")  # Expected output: (1, 3, 224, 224)


### Run the image on the Extracted Convolution Layers and Visualise them

In [ ]:
# 3, h, w
# 1, h, w
# 4, visualzie

# 64, h/100, w/1000

In [ ]:
import matplotlib.pyplot as plt

# Set the model to evaluation mode (disables dropout, batch norm updates)
model.eval()

# Pass the image through the first convolutional layer and store the output
outputs = [conv_layers[0](image)]

# Iterate through the rest of the convolutional layers and pass the feature maps sequentially
for i in range(1, len(conv_layers)):
    outputs.append(conv_layers[i](outputs[-1]))
    # Each new layer takes the output of the previous layer as input

# Visualize the feature maps for each convolutional layer
for num_layer in range(len(outputs)):

    plt.figure(figsize=(50, 10))  # Create a figure for visualization
    layer_viz = outputs[num_layer][0, :, :, :]  # Extract the feature maps (ignore batch dimension)
    layer_viz = layer_viz.data  # Detach from computation graph to convert to NumPy

    print("Layer", num_layer + 1)  # Print the layer number being visualized

    # Iterate over the first 16 filters in the current layer and visualize them
    for i, filter in enumerate(layer_viz):
        if i == 16:  # Limit visualization to the first 16 filters
            break
        plt.subplot(2, 8, i + 1)  # Arrange subplots (2 rows, 8 columns)
        plt.imshow(filter, cmap='gray')  # Display the filter response in grayscale
        plt.axis("off")  # Hide axis for cleaner visualization

    plt.show()  # Show the visualization
    plt.close()  # Close the figure to free up memory


## ResNet

### Imports

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import cv2 as cv
from torchvision import models, transforms

### Load the model

In [ ]:
import torchvision.models as models

# Load the pre-trained ResNet-50 model
model = models.resnet50(pretrained=True)

# Print the architecture of the ResNet-50 model
print(model)


### Extracting the Convolution Layers from ResNet

In [ ]:
# List to store the weights of all convolutional layers
model_weights = []

# List to store the actual convolutional layers (Conv2d layers)
conv_layers = []

# Get all the model's child modules (layers/blocks) as a list
model_children = list(model.children())
# - ResNet-50 consists of multiple layers including convolutional, batch normalization, pooling, residual blocks, and fully connected layers.
# - `list(model.children())` extracts all top-level components of the model.


In [ ]:
# Counter to keep track of the number of convolutional layers
counter = 0

# Iterate through all the top-level child modules of the model
for i in range(len(model_children)):

    # Check if the module is a Conv2D layer
    if isinstance(model_children[i], nn.Conv2d):
        counter += 1  # Increment convolutional layer count
        model_weights.append(model_children[i].weight)  # Store the layer's weights
        conv_layers.append(model_children[i])  # Store the Conv2D layer

    # If the module is a Sequential block (e.g., residual blocks in ResNet)
    elif isinstance(model_children[i], nn.Sequential):

        # Iterate through each layer inside the Sequential block
        for j in range(len(model_children[i])):

            # Iterate through the children layers inside the current residual block
            for child in model_children[i][j].children():

                # Check if the child layer is a Conv2D layer
                if isinstance(child, nn.Conv2d):
                    counter += 1  # Increment convolutional layer count
                    model_weights.append(child.weight)  # Store the layer's weights
                    conv_layers.append(child)  # Store the Conv2D layer

# Print the total number of convolutional layers found in ResNet-50
print(f"Total convolutional layers: {counter}")


In [ ]:
# take a look at the conv layers and the respective weights
for weight, conv in zip(model_weights, conv_layers):
    # print(f"WEIGHT: {weight} \nSHAPE: {weight.shape}")
    print(f"CONV: {conv} ====> SHAPE: {weight.shape}")

### Visualise the Filters

In [ ]:
import matplotlib.pyplot as plt

# Visualize the first convolutional layer filters
plt.figure(figsize=(20, 17))  # Set figure size for better visibility

# Iterate over the filters in the first convolutional layer
for i, filter in enumerate(model_weights[0]):
    plt.subplot(8, 8, i+1)  # Create an 8x8 grid to visualize 64 filters
    # - The first convolutional layer in ResNet-50 has 64 filters of size 7x7 (as per model architecture).

    plt.imshow(filter[0, :, :].detach(), cmap='gray')
    # - `filter[0, :, :]` selects the first channel of the filter (since it's 2D).
    # - `detach()` removes the tensor from computation graph (ensuring no gradient tracking).
    # - `cmap='gray'` ensures the filters are displayed in grayscale.

    plt.axis('off')  # Remove axis labels for cleaner visualization

    plt.savefig('/content/filter.png')  # Save the visualized filters as an image file

plt.show()  # Display the filters


In [ ]:
# Pass the image through all the convolutional layers

# Store the output of the first convolutional layer
results = [conv_layers[0](image)]

# Iterate through the remaining convolutional layers
for i in range(1, len(conv_layers)):
    # Pass the output from the previous layer as input to the next layer
    results.append(conv_layers[i](results[-1]))

# Make a copy of the results for further processing or visualization
outputs = results.copy()


### Run the image on the Extracted Convolution Layers and Visualise them

In [ ]:
import matplotlib.pyplot as plt

# Visualize 64 feature maps from each convolutional layer
# (Upper layers have more feature maps, but we limit it to 64 for visualization)
for num_layer in range(len(outputs)):

    plt.figure(figsize=(30, 30))  # Set figure size for better visibility

    layer_viz = outputs[num_layer][0, :, :, :]  # Extract feature maps for the current layer
    layer_viz = layer_viz.data  # Detach from computation graph (ensuring no gradient tracking)

    print(layer_viz.size())  # Print the size of the feature maps for debugging

    # Iterate through the first 64 feature maps from the current layer
    for i, filter in enumerate(layer_viz):
        if i == 64:  # Limit visualization to an 8x8 grid (64 filters max)
            break

        plt.subplot(8, 8, i + 1)  # Arrange filters in an 8x8 grid
        plt.imshow(filter, cmap='gray')  # Display feature maps in grayscale
        plt.axis("off")  # Hide axis for cleaner visualization

    print(f"Saving layer {num_layer} feature maps...")  # Print progress message

    # Uncomment the line below to save each layer's visualization as an image file
    # plt.savefig(f"/content/layer_{num_layer}.png")

    plt.show()  # Display the feature maps
    plt.close()  # Close the figure to free up memory
